In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
        .appName("spark-local-app")
        .master("local[*]")

        .config("spark.hadoop.fs.s3a.access.key", "admin")
        .config("spark.hadoop.fs.s3a.secret.key", "password123")

        .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

        .getOrCreate()
)

In [ ]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9000",
    aws_access_key_id="ROOTUSER",
    aws_secret_access_key="CHANGEME123",
    region_name="us-east-1"
)


print(s3.list_buckets())

In [ ]:
student_directory = "t_bekov"

df_orders_json = spark.read.json("s3a://kc-hardda-projects/shared/orders.json")
df_orders_json.show(3)

df_orders_json.json.coalesce(1).write.parquet(f"s3a://kc-hardda-projects/{student_directory}/orders_parquet",mode="overwrite")

In [ ]:
df_orders_parquet = spark.read.parquet(f"s3a://kc-hardda-projects/{student_directory}/orders_parquet")
df_orders_parquet.show(3)

In [ ]:
schema = """
    order_id INT,
    order_date TIMESTAMP,
    order_customer_id INT,
    order_status STRING
"""

spark.read.schema(schema).csv("s3a://kc-hardda-projects/shared/orders.csv").show(3)

In [ ]:
df_csv = spark.read.csv("s3a://kc-hardda-projects/shared/orders.csv", schema=schema)
df_csv.show(3)

In [ ]:
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("order_date", TimestampType()),
    StructField("order_customer_id", IntegerType()),
    StructField("order_status", StringType())
])

df_csv = spark.read.csv("s3a://kc-hardda-projects/shared/orders.csv", schema=schema)
df_csv.show(3)

In [ ]:
df_csv = spark.read.option("inferSchema", True) \
    .csv("s3a://kc-hardda-projects/shared/orders.csv") \
    .toDF("order_id", "order_date", "order_customer_id", "order_status")

df_csv.show()
df_csv.count()

In [ ]:
columns = ["order_id", "order_date", "order_customer_id", "order_status"]

df_csv = spark.read.option("inferSchema", True) \
    .csv("s3a://kc-hardda-projects/shared/orders.csv") \
    .toDF(*columns)

df_csv.show(3)

In [ ]:
spark.read.csv("s3a://kc-hardda-projects/shared/orders.csv", sep=",", header=None, schema=schema).show()

In [29]:
help(spark.read.csv)

Help on method csv in module pyspark.sql.readwriter:

csv(
    path: Union[str, List[str], 'RDD[str]', 'DataFrame'],
    schema: Optional[Union[StructType, str]] = None,
    sep: Optional[str] = None,
    encoding: Optional[str] = None,
    quote: Optional[str] = None,
    escape: Optional[str] = None,
    comment: Optional[str] = None,
    header: Optional[Union[bool, str]] = None,
    inferSchema: Optional[Union[bool, str]] = None,
    ignoreLeadingWhiteSpace: Optional[Union[bool, str]] = None,
    ignoreTrailingWhiteSpace: Optional[Union[bool, str]] = None,
    nullValue: Optional[str] = None,
    nanValue: Optional[str] = None,
    positiveInf: Optional[str] = None,
    negativeInf: Optional[str] = None,
    dateFormat: Optional[str] = None,
    timestampFormat: Optional[str] = None,
    maxColumns: Optional[Union[int, str]] = None,
    maxCharsPerColumn: Optional[Union[int, str]] = None,
    maxMalformedLogPerPartition: Optional[Union[int, str]] = None,
    mode: Optional[str] = N

In [ ]:
df_csv = spark. \
    read. \
    options(sep=",", header=None, schema=schema). \
    csv("s3a://kc-hardda-projects/shared/orders.csv"). \
    toDF(*columns)

df_csv.show(3)